In [1]:
!pip install espnet espnet_model_zoo onnxscript onnxruntime --quiet

import espnet2
print(f"espnet2: {espnet2.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 12.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 14.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 58.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 87.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━

In [28]:
import os
BASE = "/kaggle/working"
RAW  = "https://github.com/smtiitm/Fastspeech2_HS/raw/main"

os.system(f'wget -q "{RAW}/multilingualcharmap.json" -O "{BASE}/multilingualcharmap.json"')

size = os.path.getsize(f"{BASE}/multilingualcharmap.json")
print(f"multilingualcharmap.json: {size/1024:.1f} KB")

multilingualcharmap.json: 23.4 KB


In [2]:
import os

BASE = "/kaggle/working"
RAW  = "https://github.com/smtiitm/Fastspeech2_HS/raw/main"

os.makedirs(f"{BASE}/hifigan", exist_ok=True)
os.makedirs(f"{BASE}/vocoder/female/aryan/hifigan", exist_ok=True)

files = {
    f"{BASE}/hifigan/models.py"                      : f"{RAW}/hifigan/models.py",
    f"{BASE}/hifigan/env.py"                         : f"{RAW}/hifigan/env.py",
    f"{BASE}/hifigan/meldataset.py"                  : f"{RAW}/hifigan/meldataset.py",
    f"{BASE}/text_preprocess_for_inference.py"       : f"{RAW}/text_preprocess_for_inference.py",
    f"{BASE}/vocoder/female/aryan/hifigan/generator" : f"{RAW}/vocoder/female/aryan/hifigan/generator",
}

for dest, url in files.items():
    fname = os.path.basename(dest)
    print(f"Downloading {fname}...")
    os.system(f'wget -q "{url}" -O "{dest}"')
    size = os.path.getsize(dest) if os.path.exists(dest) else 0
    print(f"  {fname}: {size/1024:.1f} KB")

  models.py: 9.7 KB
  env.py: 0.4 KB
  meldataset.py: 6.2 KB
  text_preprocess_for_inference.py: 34.8 KB
  generator: 54520.7 KB


In [16]:
import subprocess, os, yaml

BASE      = "/kaggle/working"
MODEL_RAW = "https://github.com/smtiitm/Fastspeech2_HS/raw/main/gujarati/female/model"
MODEL_DIR = f"{BASE}/gujarati/female/model"
os.makedirs(MODEL_DIR, exist_ok=True)

for fname in ["config.yaml", "energy_stats.npz", "feats_stats.npz",
              "feats_type", "model.pth", "pitch_stats.npz"]:
    url  = f"{MODEL_RAW}/{fname}"
    dest = f"{MODEL_DIR}/{fname}"
    print(f"Downloading {fname}...")
    subprocess.run(["wget", "-q", url, "-O", dest])
    size = os.path.getsize(dest) if os.path.exists(dest) else 0
    print(f"  {fname}: {size/1024:.1f} KB")

# Patch all stats_file paths
with open(f"{MODEL_DIR}/config.yaml") as f:
    cfg = yaml.safe_load(f)

patched = 0
for key, val in cfg.items():
    if isinstance(val, dict) and "stats_file" in val:
        fname = os.path.basename(val["stats_file"])
        cfg[key]["stats_file"] = f"{MODEL_DIR}/{fname}"
        print(f"Patched {key} → {cfg[key]['stats_file']}")
        patched += 1

with open(f"{MODEL_DIR}/config.yaml", "w") as f:
    yaml.dump(cfg, f)
print(f"\nTotal patched: {patched}  ✅")

  config.yaml: 5.3 KB
  energy_stats.npz: 0.8 KB
  feats_stats.npz: 1.4 KB
  feats_type: 0.0 KB
  model.pth: 145203.2 KB
  pitch_stats.npz: 0.8 KB
Patched normalize_conf → /kaggle/working/gujarati/female/model/feats_stats.npz
Patched pitch_normalize_conf → /kaggle/working/gujarati/female/model/pitch_stats.npz
Patched energy_normalize_conf → /kaggle/working/gujarati/female/model/energy_stats.npz

Total patched: 3  ✅


In [3]:
utils_code = '''
import os
import torch
from torch.nn.utils import weight_norm

def init_weights(m, mean=0.0, std=0.01):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        m.weight.data.normal_(mean, std)

def apply_weight_norm(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        weight_norm(m)

def get_padding(kernel_size, dilation=1):
    return int((kernel_size * dilation - dilation) / 2)

def load_checkpoint(filepath, device):
    assert os.path.isfile(filepath)
    print(f"Loading '{filepath}'")
    checkpoint_dict = torch.load(filepath, map_location=device)
    print("Complete.")
    return checkpoint_dict

def save_checkpoint(filepath, obj):
    print(f"Saving checkpoint to {filepath}")
    torch.save(obj, filepath)
    print("Complete.")

def scan_checkpoint(cp_dir, prefix):
    pattern = os.path.join(cp_dir, prefix + "????????")
    import glob
    cp_list = glob.glob(pattern)
    if len(cp_list) == 0:
        return None
    return sorted(cp_list)[-1]
'''

with open(f"{BASE}/hifigan/utils.py", "w") as f:
    f.write(utils_code)

print("utils.py written.")

utils.py written.


In [4]:
import subprocess

MODEL_RAW = "https://github.com/smtiitm/Fastspeech2_HS/raw/main/gujarati/female/model"
MODEL_DIR = f"{BASE}/gujarati/female/model"
os.makedirs(MODEL_DIR, exist_ok=True)

model_files = ["config.yaml", "energy_stats.npz", "feats_stats.npz",
               "feats_type", "model.pth", "pitch_stats.npz"]

for fname in model_files:
    url  = f"{MODEL_RAW}/{fname}"
    dest = f"{MODEL_DIR}/{fname}"
    print(f"Downloading {fname}...")
    subprocess.run(["wget", "-q", url, "-O", dest])
    size = os.path.getsize(dest) if os.path.exists(dest) else 0
    print(f"  {fname}: {size/1024:.1f} KB")

  config.yaml: 5.3 KB
  energy_stats.npz: 0.8 KB
  feats_stats.npz: 1.4 KB
  feats_type: 0.0 KB
  model.pth: 145203.2 KB
  pitch_stats.npz: 0.8 KB


In [17]:
import sys, os, torch
from espnet2.bin.tts_inference import Text2Speech

BASE      = "/kaggle/working"
MODEL_DIR = f"{BASE}/gujarati/female/model"

sys.path.insert(0, BASE)

tts = Text2Speech(
    train_config = f"{MODEL_DIR}/config.yaml",
    model_file   = f"{MODEL_DIR}/model.pth",
    device       = "cpu",
)
tts.model.eval()
print("FastSpeech2 female model loaded ✅")

FastSpeech2 female model loaded ✅


In [15]:
import json

BASE = "/kaggle/working"

config = {
    "resblock": "1", "num_gpus": 0, "batch_size": 16,
    "learning_rate": 0.0002, "adam_b1": 0.8, "adam_b2": 0.99,
    "lr_decay": 0.999, "seed": 1234,
    "upsample_rates": [8, 8, 2, 2],
    "upsample_kernel_sizes": [16, 16, 4, 4],
    "upsample_initial_channel": 512,
    "resblock_kernel_sizes": [3, 7, 11],
    "resblock_dilation_sizes": [[1,3,5],[1,3,5],[1,3,5]],
    "segment_size": 8192, "num_mels": 80, "num_freq": 1025,
    "n_fft": 1024, "hop_size": 256, "win_size": 1024,
    "sampling_rate": 22050, "fmin": 0, "fmax": 8000,
    "fmax_for_loss": None, "num_workers": 4,
    "dist_config": {
        "dist_backend": "nccl",
        "dist_url": "tcp://localhost:54321",
        "world_size": 1
    }
}

with open(f"{BASE}/vocoder/female/aryan/hifigan/config.json", "w") as f:
    json.dump(config, f, indent=4)
print("config.json written ✅")

config.json written ✅


In [18]:
import torch, os, shutil

BASE       = "/kaggle/working"
MODEL_DIR  = f"{BASE}/gujarati/female/model"
FS2_QU_DIR = f"{BASE}/FS2-qu-female"
os.makedirs(FS2_QU_DIR, exist_ok=True)

from espnet2.bin.tts_inference import Text2Speech

tts = Text2Speech(
    train_config = f"{MODEL_DIR}/config.yaml",
    model_file   = f"{MODEL_DIR}/model.pth",
    device       = "cpu",
)

tts.model = tts.model.half()

torch.save(tts.model.state_dict(), f"{FS2_QU_DIR}/model_fp16.pt")
shutil.copy(f"{MODEL_DIR}/config.yaml", f"{FS2_QU_DIR}/config.yaml")

orig = os.path.getsize(f"{MODEL_DIR}/model.pth") / 1e6
fp16 = os.path.getsize(f"{FS2_QU_DIR}/model_fp16.pt") / 1e6
print(f"Original : {orig:.2f} MB")
print(f"FP16     : {fp16:.2f} MB  ({(1-fp16/orig)*100:.1f}% smaller) ✅")

Original : 148.69 MB
FP16     : 74.39 MB  (50.0% smaller) ✅


In [19]:
import torch, numpy as np, sys, os
BASE        = "/kaggle/working"
MODEL_DIR   = f"{BASE}/gujarati/female/model"
FS2_QU_DIR  = f"{BASE}/FS2-qu-female"
EXPORT_DIR  = f"{BASE}/exports-female"
os.makedirs(EXPORT_DIR, exist_ok=True)
sys.path.insert(0, BASE)

from espnet2.bin.tts_inference import Text2Speech
import espnet.nets.pytorch_backend.fastspeech.length_regulator as lr_module

tts = Text2Speech(
    train_config = f"{FS2_QU_DIR}/config.yaml",
    model_file   = f"{FS2_QU_DIR}/model_fp16.pt",
    device       = "cpu",
)
tts.model = tts.model.float()
tts.model.eval()
print("✅ FastSpeech2 female FP16 loaded")

# ── Patch LengthRegulator so ONNX export works ────────────────────────────────
def patched_forward(self, xs, ds, alpha=1.0):
    ds = ds.long()
    output = torch.cat([
        xs[i].repeat_interleave(ds[i], dim=0)
        for i in range(xs.size(0))
    ], dim=0).unsqueeze(0)
    return output

lr_module.LengthRegulator.forward = patched_forward
print("✅ LengthRegulator patched")

fs2 = tts.model.tts

# ── Encoder wrapper ───────────────────────────────────────────────────────────
class FS2EncoderWrapper(torch.nn.Module):
    def __init__(self, fs2):
        super().__init__()
        self.encoder            = fs2.encoder
        self.duration_predictor = fs2.duration_predictor
        self.pitch_predictor    = fs2.pitch_predictor
        self.energy_predictor   = fs2.energy_predictor
        self.pitch_embed        = fs2.pitch_embed
        self.energy_embed       = fs2.energy_embed

    def forward(self, text_ids: torch.Tensor):
        hs, _  = self.encoder(text_ids, None)
        d_outs = self.duration_predictor(hs, None)
        p_outs = self.pitch_predictor(hs, None)
        p_emb  = self.pitch_embed(p_outs.transpose(1,2)).transpose(1,2)
        hs     = hs + p_emb
        e_outs = self.energy_predictor(hs, None)
        e_emb  = self.energy_embed(e_outs.transpose(1,2)).transpose(1,2)
        hs     = hs + e_emb
        return hs, d_outs  # [1,T,384], [1,T] log durations

# ── Decoder wrapper ───────────────────────────────────────────────────────────
class FS2DecoderWrapper(torch.nn.Module):
    def __init__(self, fs2):
        super().__init__()
        self.decoder  = fs2.decoder
        self.feat_out = fs2.feat_out
        self.postnet  = fs2.postnet

    def forward(self, hs: torch.Tensor):
        zs, _  = self.decoder(hs, None)
        before = self.feat_out(zs)
        after  = before + self.postnet(before.transpose(1,2)).transpose(1,2)
        return after.transpose(1,2)  # [1,80,T']

enc_wrapper = FS2EncoderWrapper(fs2).eval().float()
dec_wrapper = FS2DecoderWrapper(fs2).eval().float()

dummy_ids = torch.randint(1, 50, (1, 20), dtype=torch.long)

# Verify
with torch.no_grad():
    hs, d = enc_wrapper(dummy_ids)
    durations = torch.clamp(torch.round(d), min=0).long()
    hs_exp = torch.cat([
        hs[0][i].unsqueeze(0).repeat(durations[0][i].item(), 1)
        for i in range(hs.size(1))
    ], dim=0).unsqueeze(0)
    mel = dec_wrapper(hs_exp)
print(f"✅ Forward pass OK — hs:{hs.shape}, mel:{mel.shape}")

# ── Export encoder ────────────────────────────────────────────────────────────
ENC_ONNX = f"{EXPORT_DIR}/fs2_encoder_female.onnx"
with torch.no_grad():
    torch.onnx.export(
        enc_wrapper, (dummy_ids,), ENC_ONNX,
        input_names   = ["text_ids"],
        output_names  = ["hidden_states", "durations"],
        dynamic_axes  = {
            "text_ids"      : {1: "text_len"},
            "hidden_states" : {1: "text_len"},
            "durations"     : {1: "text_len"},
        },
        opset_version       = 17,
        do_constant_folding = False,
        dynamo              = False,
    )
print(f"✅ Encoder ONNX → {ENC_ONNX}")

# ── Export decoder ────────────────────────────────────────────────────────────
DEC_ONNX = f"{EXPORT_DIR}/fs2_decoder_female.onnx"
dummy_hs  = torch.randn(1, 100, 384)
with torch.no_grad():
    torch.onnx.export(
        dec_wrapper, (dummy_hs,), DEC_ONNX,
        input_names   = ["hidden_states"],
        output_names  = ["mel_spectrogram"],
        dynamic_axes  = {
            "hidden_states"  : {1: "mel_len"},
            "mel_spectrogram": {2: "mel_len"},
        },
        opset_version       = 17,
        do_constant_folding = False,
        dynamo              = False,
    )
print(f"✅ Decoder ONNX → {DEC_ONNX}")

import onnx
onnx.checker.check_model(ENC_ONNX)
onnx.checker.check_model(DEC_ONNX)
print("✅ Both models verified")

✅ FastSpeech2 female FP16 loaded
✅ LengthRegulator patched
✅ Forward pass OK — hs:torch.Size([1, 20, 384]), mel:torch.Size([1, 80, 57])


/tmp/ipykernel_55/1359318960.py:90: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/espnet/nets/pytorch_backend/transformer/embedding.py:65: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if self.pe.size(1) >= x.size(1):


✅ Encoder ONNX → /kaggle/working/exports-female/fs2_encoder_female.onnx


/tmp/ipykernel_55/1359318960.py:109: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


✅ Decoder ONNX → /kaggle/working/exports-female/fs2_decoder_female.onnx
✅ Both models verified


In [20]:
import torch, os, sys, json
import onnx, onnxruntime as ort

BASE        = "/kaggle/working"
VOCODER_DIR = f"{BASE}/vocoder/female/aryan/hifigan"
EXPORT_DIR  = f"{BASE}/exports-female"
sys.path.insert(0, f"{BASE}/hifigan")
sys.path.insert(0, BASE)

for mod in ["models", "env", "meldataset"]:
    sys.modules.pop(mod, None)

from models import Generator
from env import AttrDict

with open(f"{VOCODER_DIR}/config.json") as f:
    h = AttrDict(json.load(f))

torch.manual_seed(h.seed)
vocoder = Generator(h)
vocoder.load_state_dict(
    torch.load(f"{VOCODER_DIR}/generator", map_location="cpu")["generator"]
)
vocoder.eval()
vocoder.remove_weight_norm()
vocoder = vocoder.float()
print("✅ HiFi-GAN female loaded")

dummy_mel  = torch.randn(1, 80, 200)
HIFI_ONNX  = f"{EXPORT_DIR}/hifigan_female.onnx"

with torch.no_grad():
    traced_voc = torch.jit.trace(vocoder, dummy_mel, check_trace=False)
    torch.onnx.export(
        traced_voc, dummy_mel, HIFI_ONNX,
        input_names         = ["mel_spectrogram"],
        output_names        = ["waveform"],
        dynamic_axes        = {
            "mel_spectrogram" : {2: "mel_len"},
            "waveform"        : {2: "samples"},
        },
        opset_version       = 17,
        do_constant_folding = False,
        dynamo              = False,
    )

size = os.path.getsize(HIFI_ONNX) / 1e6
print(f"✅ HiFi-GAN ONNX → {HIFI_ONNX}  ({size:.1f} MB)")

onnx.checker.check_model(HIFI_ONNX)
sess = ort.InferenceSession(HIFI_ONNX)
out  = sess.run(None, {"mel_spectrogram": dummy_mel.numpy()})
print(f"✅ Verified — waveform shape: {out[0].shape}")

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Removing weight norm...
✅ HiFi-GAN female loaded


/tmp/ipykernel_55/698552032.py:34: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1508: UserWarning: no signature found for builtin <built-in method __call__ of pybind11_builtins.pybind11_detail_function_record_v1_system_libstdcpp_gxx_abi_1xxx_use_cxx11_abi_1 object at 0x793b75ad8a10>, skipping _decide_input_format
  args = _decide_input_format(model, args)


✅ HiFi-GAN ONNX → /kaggle/working/exports-female/hifigan_female.onnx  (55.7 MB)
✅ Verified — waveform shape: (1, 1, 51200)


In [21]:
from onnxruntime.quantization import quantize_dynamic, QuantType
import os

EXPORT_DIR = "/kaggle/working/exports-female"

quantize_dynamic(
    model_input  = f"{EXPORT_DIR}/fs2_encoder_female.onnx",
    model_output = f"{EXPORT_DIR}/fs2_encoder_female_int8.onnx",
    weight_type  = QuantType.QInt8,
)
print(f"✅ Encoder INT8: {os.path.getsize(f'{EXPORT_DIR}/fs2_encoder_female_int8.onnx')/1e6:.1f} MB")

quantize_dynamic(
    model_input  = f"{EXPORT_DIR}/fs2_decoder_female.onnx",
    model_output = f"{EXPORT_DIR}/fs2_decoder_female_int8.onnx",
    weight_type  = QuantType.QInt8,
)
print(f"✅ Decoder INT8: {os.path.getsize(f'{EXPORT_DIR}/fs2_decoder_female_int8.onnx')/1e6:.1f} MB")

quantize_dynamic(
    model_input  = f"{EXPORT_DIR}/hifigan_female.onnx",
    model_output = f"{EXPORT_DIR}/hifigan_female_int8.onnx",
    weight_type  = QuantType.QInt8,
)
print(f"✅ HiFi-GAN INT8: {os.path.getsize(f'{EXPORT_DIR}/hifigan_female_int8.onnx')/1e6:.1f} MB")

✅ Encoder INT8: 34.3 MB


✅ Decoder INT8: 32.8 MB
✅ HiFi-GAN INT8: 22.1 MB


In [23]:
import sys
from unittest.mock import MagicMock

sys.modules['num_to_words'] = MagicMock()
sys.modules['NumberToText'] = MagicMock()

# Clear stale cached copy so the fresh import picks up the mock
sys.modules.pop('text_preprocess_for_inference', None)

print("Mocked missing modules")

Mocked missing modules


In [25]:
import os
BASE = "/kaggle/working"
RAW  = "https://github.com/smtiitm/Fastspeech2_HS/raw/main"

os.makedirs(f"{BASE}/phone_dict", exist_ok=True)

!wget -q "{RAW}/phone_dict/gujarati" -O "{BASE}/phone_dict/gujarati"

size = os.path.getsize(f"{BASE}/phone_dict/gujarati")
print(f" phone_dict/gujarati: {size/1024:.1f} KB")

 phone_dict/gujarati: 3509.2 KB


In [29]:
import numpy as np, sys, os, yaml
import onnxruntime as ort
from scipy.io.wavfile import write as wav_write
from IPython.display import Audio, display

BASE        = "/kaggle/working"
EXPORT_DIR  = f"{BASE}/exports-female"
MODEL_DIR   = f"{BASE}/gujarati/female/model"
SAMPLING_RATE = 22050
MAX_WAV_VALUE = 32768.0
sys.path.insert(0, BASE)

from text_preprocess_for_inference import TTSDurAlignPreprocessor

# Use INT8 models (swap to non-_int8 filenames if you want fp32 ONNX instead)
sess_enc  = ort.InferenceSession(f"{EXPORT_DIR}/fs2_encoder_female_int8.onnx", providers=["CPUExecutionProvider"])
sess_dec  = ort.InferenceSession(f"{EXPORT_DIR}/fs2_decoder_female_int8.onnx", providers=["CPUExecutionProvider"])
sess_hifi = ort.InferenceSession(f"{EXPORT_DIR}/hifigan_female_int8.onnx",     providers=["CPUExecutionProvider"])
print("✅ All 3 ONNX models loaded — fully ONNX, no PyTorch")

# Build token2id from config
with open(f"{MODEL_DIR}/config.yaml") as f:
    config = yaml.safe_load(f)
token_list = config["token_list"]
token2id   = {tok: idx for idx, tok in enumerate(token_list)}
print(f"✅ Vocab: {len(token_list)} tokens")

# Mel stats
stats    = np.load(f"{MODEL_DIR}/feats_stats.npz")
count    = stats["count"]
mel_mean = (stats["sum"] / count).astype(np.float32)
mel_std  = (np.sqrt(stats["sum_square"] / count - mel_mean ** 2)).astype(np.float32)
print("✅ Mel stats loaded")

preprocessor = TTSDurAlignPreprocessor()
print("✅ Ready\n")

def synthesize_onnx(text, label="test"):
    print(f"🔊 [{label}] {text[:60]}")

    preprocessed_list, _ = preprocessor.preprocess(text, "gujarati", "female")
    preprocessed_text    = " ".join(preprocessed_list)
    print(f"  Processed : {preprocessed_text[:70]}")

    # Text → token IDs (char by char, skip spaces)
    token_ids = np.array(
        [[token2id.get(c, 1) for c in preprocessed_text if c != ' ']],
        dtype=np.int64
    )
    print(f"  Token IDs : {token_ids.shape}")

    # Step 1: Encoder → hidden states + log durations
    hs, d_log = sess_enc.run(None, {"text_ids": token_ids})

    # Step 2: exp + round → frame counts
    durations = np.clip(np.round(np.exp(d_log[0])), 0, None).astype(int)
    print(f"  Durations : total={durations.sum()}")

    # Step 3: Length regulate in Python
    hs_expanded = np.concatenate(
        [np.repeat(hs[0][i:i+1], durations[i], axis=0) for i in range(hs.shape[1])],
        axis=0
    )[np.newaxis, :].astype(np.float32)

    # Step 4: Decoder → mel
    mel_norm   = sess_dec.run(None, {"hidden_states": hs_expanded})[0]

    # Step 5: Denorm + scale
    mel_denorm = (mel_norm * mel_std[None,:,None] + mel_mean[None,:,None]).astype(np.float32)
    mel_scaled = (mel_denorm * 2.3262).astype(np.float32)

    # Step 6: HiFi-GAN → waveform
    wav   = sess_hifi.run(None, {"mel_spectrogram": mel_scaled})[0].squeeze()
    wav   = wav / (np.abs(wav).max() + 1e-8)
    audio = (wav * MAX_WAV_VALUE).astype(np.int16)

    path = f"{BASE}/output/female_onnx_{label}.wav"
    os.makedirs(f"{BASE}/output", exist_ok=True)
    wav_write(path, SAMPLING_RATE, audio)
    print(f"  {len(audio)/SAMPLING_RATE:.2f}s → {path}")
    display(Audio(path, rate=SAMPLING_RATE))

synthesize_onnx("નમસ્તે, તમે કેમ છો?",                                          label="short")
synthesize_onnx("ભારત એક મહાન દેશ છે અને અહીં અનેક ભાષાઓ બોલવામાં આવે છે.", label="medium")
synthesize_onnx("આજે હવામાન ખૂબ સારું છે. બાળકો બગીચામાં રમી રહ્યા છે.",     label="long")

Phone dictionary loaded for the following languages: ['gujarati']
Loading G2P model... Done!
Phone dictionary loaded for the following languages: ['gujarati']
Loading G2P model... Done!
Phone dictionary loaded for the following languages: ['gujarati']
Loading G2P model... Done!
Phone dictionary loaded for the following languages: ['gujarati']
Loading G2P model... Done!
Phone dictionary loaded for the following languages: ['gujarati']
Loading G2P model... Done!
✅ All 3 ONNX models loaded — fully ONNX, no PyTorch
✅ Vocab: 56 tokens
✅ Mel stats loaded
✅ Ready

🔊 [short] નમસ્તે, તમે કેમ છો?
નમસ્તે, તમે કેમ છો?
cleaned text નમસ્તે#  તમે કેમ છો
word not in dict: []
['$namastE.', '$tamEkEmCo.']
  Processed : $namastE. $tamEkEmCo.
  Token IDs : (1, 20)
  Durations : total=291
  ✅ 3.38s → /kaggle/working/output/female_onnx_short.wav


🔊 [medium] ભારત એક મહાન દેશ છે અને અહીં અનેક ભાષાઓ બોલવામાં આવે છે.
ભારત એક મહાન દેશ છે અને અહીં અનેક ભાષાઓ બોલવામાં આવે છે.
cleaned text ભારત એક મહાન દેશ છે અને અહીં અનેક ભાષાઓ બોલવામાં આવે છે# 
word not in dict: []
['$BAratEkmahAndEशCEanEahIqanEkBAषAobolwAmAqAwECE.']
  Processed : $BAratEkmahAndEशCEanEahIqanEkBAषAobolwAmAqAwECE.
  Token IDs : (1, 48)
  Durations : total=543
  ✅ 6.30s → /kaggle/working/output/female_onnx_medium.wav


🔊 [long] આજે હવામાન ખૂબ સારું છે. બાળકો બગીચામાં રમી રહ્યા છે.
આજે હવામાન ખૂબ સારું છે. બાળકો બગીચામાં રમી રહ્યા છે.
cleaned text આજે હવામાન ખૂબ સારું છે#  બાળકો બગીચામાં રમી રહ્યા છે# 
word not in dict: []
['$AjEhawAmAnखUbsAruqCE.', '$bAളkobagIcAmAqramIrahyACE.']
  Processed : $AjEhawAmAnखUbsAruqCE. $bAളkobagIcAmAqramIrahyACE.
  Token IDs : (1, 49)
  Durations : total=632
  ✅ 7.34s → /kaggle/working/output/female_onnx_long.wav
